In [1]:
# Setup (standalone run): load splits + rebuild preprocessor
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train = pd.read_csv("data/X_train.csv")
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)
print("Setup done:", X_train.shape)

Setup done: (5634, 18)


## Task 4: Model Selection

**Candidates:**
- (A) `LogisticRegression` — baseline: fast, interpretable, works well with scaled one-hot features; `class_weight='balanced'` counters the 73/27 imbalance; `max_iter=1000` ensures convergence on 45 features.
- (B) `RandomForestClassifier` — comparator: captures non-linear interactions found in Task 3 (e.g. Contract x tenure x Fiber optic) without manual feature engineering; `class_weight='balanced'` for imbalance; `n_estimators=200` for stability.

**Why these two:**
- Dataset is small (5634 train rows x 45 encoded cols) — both train in seconds, no need for heavy gradient boosting yet.
- Logistic regression gives explainable coefficients (useful for business recommendations); random forest tests whether interactions add value.
- Both are wrapped in a `Pipeline(preprocessor + model)` so encoding/scaling is refit on train only — no leakage.
- Decision (best of the two) will be made in Task 6 using accuracy, precision, recall, F1 and ROC-AUC.

In [2]:
# Task 4: define candidate models as leakage-safe pipelines (training happens in Task 5)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pipe_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)),
])

print("Baseline (A):", pipe_lr.named_steps["model"])
print("Comparator (B):", pipe_rf.named_steps["model"])

Baseline (A): LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
Comparator (B): RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)
